## 猜數字遊戲-文字版
本專案是一個基於命令列介面 (CLI) 的經典猜數字遊戲。程式會隨機生成一個介於 1 到 100 之間的目標數字，玩家的目標是在限定的猜測次數內猜中該數字。本遊戲最大的特色是提供了兩種難度模式，以調整遊戲的挑戰性。

![upgit_20260323_1774256048.png|638x305](https://raw.githubusercontent.com/kcwc1029/obsidian-upgit-image/main/2026/03/upgit_20260323_1774256048.png)


In [1]:
import random

In [2]:
# 設置難度模式和允許的猜測次數
EASY_LEVEL_TURNS = 10
HARD_LEVEL_TURNS = 5

In [3]:
def set_difficulty():
    """
	讓玩家選擇遊戲難度並設定猜測次數。
	"""
    while True:
        level = input("請選擇一個難度。輸入 'easy' (簡單) 或 'hard' (困難): ").lower()
        if level == "easy":
            return EASY_LEVEL_TURNS
        elif level == "hard":
            return HARD_LEVEL_TURNS
        else:
            print("輸入無效，請重新輸入。")

In [4]:
def check_guess(guess, answer, turns):
    """
	檢查猜測的數字是否正確，並返回剩餘的猜測次數
	"""
    if guess > answer:
        print("太高了。")
        return turns - 1
    elif guess < answer:
        print("太低了。")
        return turns - 1
    else:
        print(f"你猜對了！答案是 {answer}。")
        return 0 # 猜對了，剩餘次數歸零以結束遊戲

In [5]:
### 遊戲主流程
print("猜數字遊戲")
print("我正在想一個 1 到 100 之間的數字。")

answer = random.randint(1, 100)

# 設置難度
turns = set_difficulty()
print(f"你選擇了 {turns} 次機會來猜數字。")
# 玩家猜的數字
guess = 0

# 遊戲迴圈：直到猜對 (turns=0) 或次數用完
while guess != answer and turns > 0:
	print(f"你還剩下 {turns} 次猜測機會。")

	# 處理無效輸入
	try:
		guess = int(input("請猜一個數字: "))
	except ValueError:
		print("無效輸入，請輸入一個整數。")
		continue # 跳過本次循環，不扣除次數

	# 檢查猜測結果並更新剩餘次數
	turns = check_guess(guess, answer, turns)

	if turns > 0 and guess != answer:
		print("再猜一次。")

	elif turns == 0 and guess != answer:
		print("你所有的猜測機會都用完了。你輸了！")
		print(f"正確答案是 {answer}")

猜數字遊戲
我正在想一個 1 到 100 之間的數字。


KeyboardInterrupt: Interrupted by user

## 猜數字遊戲-增加介面版

In [8]:
!pip install gradio -q

In [9]:
import gradio as gr
import random

# 設定初始常數
EASY_LEVEL_TURNS = 10
HARD_LEVEL_TURNS = 5

def start_new_game(difficulty):
    """初始化遊戲狀態"""
    answer = random.randint(1, 100)
    turns = EASY_LEVEL_TURNS if difficulty == "簡單 (10次)" else HARD_LEVEL_TURNS
    return answer, turns, f"遊戲開始！我已經想好了一個 1 到 100 之間的數字。你有 {turns} 次機會。", turns, gr.update(interactive=True)

def handle_guess(guess, answer, turns):
    """處理玩家猜測的邏輯"""
    if answer is None:
        return turns, "請先點擊『開始新遊戲』！", turns, gr.update(interactive=False)

    if turns <= 0:
        return 0, "遊戲已結束，請重新開始。", 0, gr.update(interactive=False)

    try:
        user_guess = int(guess)
    except (TypeError, ValueError):
        return turns, "請輸入有效的整數數字！", turns, gr.update(interactive=True)

    turns -= 1

    if user_guess == answer:
        return 0, f"你猜對了！答案是 {answer}。🎉", 0, gr.update(interactive=False)
    elif turns == 0:
        return 0, f"次數用完，你輸了！正確答案是 {answer}。💀", 0, gr.update(interactive=False)
    elif user_guess > answer:
        result = "太高了。"
    else:
        result = "太低了。"

    return turns, f"{result} 還剩下 {turns} 次機會。", turns, gr.update(interactive=True)

# 建立 Gradio 介面
with gr.Blocks() as demo:
    gr.Markdown("# 🐍 Python 猜數字遊戲")
    gr.Markdown("這是一個從 CLI 轉化為 GUI 的教學範例。")

    # 使用 gr.State 儲存遊戲中間狀態
    state_answer = gr.State()
    state_turns = gr.State()

    with gr.Row():
        difficulty_input = gr.Radio(["簡單 (10次)", "困難 (5次)"], label="設定難度", value="簡單 (10次)")
        start_btn = gr.Button("開始新遊戲 / 重置")

    with gr.Column():
        msg_output = gr.Textbox(label="系統提示", interactive=False)
        guess_input = gr.Number(label="你的猜測 (1-100)", precision=0)
        submit_btn = gr.Button("提交猜測", variant="primary", interactive=False)
        turns_display = gr.Number(label="剩餘次數顯示", interactive=False)

    # 綁定按鈕事件
    start_btn.click(
        start_new_game,
        inputs=[difficulty_input],
        outputs=[state_answer, state_turns, msg_output, turns_display, submit_btn]
    )

    submit_btn.click(
        handle_guess,
        inputs=[guess_input, state_answer, state_turns],
        outputs=[state_turns, msg_output, turns_display, submit_btn]
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bfd8fe6d4512886a1c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
